# Mango vs Zara price regression notebook

This notebook uses only the exact columns in your two CSV files and compares **Men, Women, and Kids** across both brands.

It does four things:
1. Loads your two CSVs
2. Cleans and engineers features without inventing any data
3. Builds a regression model to predict price
4. Creates comparison graphs for brand, market, segment, category, materials, discounts, and savings


In [1]:
!pip -q install plotly scikit-learn statsmodels

In [5]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Uploaded files: []


In [7]:
import pandas as pd
import numpy as np
import re
import glob
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 200)

all_csvs = glob.glob("*.csv")

mango_candidates = [f for f in all_csvs if "mango" in f.lower() and "comparison" in f.lower()]
zara_candidates = [f for f in all_csvs if "zara" in f.lower() and "combined" in f.lower()]

if not mango_candidates:
    raise FileNotFoundError("I could not find the Mango CSV after upload.")
if not zara_candidates:
    raise FileNotFoundError("I could not find the Zara CSV after upload.")

mango_file = mango_candidates[0]
zara_file = zara_candidates[0]

print("Using Mango file:", mango_file)
print("Using Zara file:", zara_file)

mango = pd.read_csv(mango_file)
zara = pd.read_csv(zara_file)

print("\\nMango shape:", mango.shape)
print("Zara shape:", zara.shape)

print("\\nMango columns:")
print(mango.columns.tolist())

print("\\nZara columns:")
print(zara.columns.tolist())

Using Mango file: mango_full_comparison_by_category.csv
Using Zara file: zara_combined_men_women_kids.csv
\nMango shape: (74, 31)
Zara shape: (593, 21)
\nMango columns:
['Product_ID', 'Segment', 'Source File', 'Specific_Category', 'US_url', 'IN_url', 'name', 'colour_from_title', 'US_price_current_usd', 'US_price_original_usd', 'US_discount_pct_display', 'US_discount_pct_calc', 'US_on_sale', 'IN_price_current_inr', 'IN_price_original_inr', 'IN_price_current_usd', 'IN_price_original_usd', 'IN_discount_pct_display', 'IN_discount_pct_calc', 'IN_on_sale', 'Savings_USD', 'materials_text', 'made_in', 'US_detail_selector', 'US_composition_selector', 'IN_detail_selector', 'IN_composition_selector', 'row_collected_at_local', 'row_collected_at_utc', 'fx_rate_live_usd_inr', 'fx_rate_timestamp_utc']
\nZara columns:
['Product_ID', 'name', 'Category', 'colour_from_title', 'US_price_current_usd', 'US_price_original_usd', 'US_discount_pct_display', 'US_on_sale', 'IN_price_current_inr', 'IN_price_curren

In [8]:
material_keywords = [
    "cotton","polyester","linen","wool","leather","viscose","lyocell",
    "elastane","spandex","modal","silk","acrylic","polyamide","nylon","cashmere"
]

def clean_country(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).split("\\n")[0].strip()
    s = re.sub(r"\\s+", " ", s)
    return s if s else "Unknown"

def extract_material_features(text):
    text = "" if pd.isna(text) else str(text).lower()
    feats = {}
    for mat in material_keywords:
        vals = []
        vals += [float(x) for x in re.findall(r"(\\d+(?:\\.\\d+)?)\\s*%\\s*" + re.escape(mat), text)]
        vals += [float(x) for x in re.findall(re.escape(mat) + r"[^0-9]{0,10}(\\d+(?:\\.\\d+)?)\\s*%", text)]
        feats[f"{mat}_max_pct"] = max(vals) if vals else 0.0
        feats[f"has_{mat}"] = int(mat in text)
    feats["material_char_len"] = len(text)
    feats["material_piece_count"] = len([p for p in re.split(r"[|,;/]+", text) if p.strip()])
    feats["mentions_composition"] = int("composition" in text)
    return feats

def prep_brand_df(df, brand):
    d = df.copy()
    d["Brand"] = brand
    d["Segment"] = d["Segment"].astype(str).str.title().replace({"Nan": "Unknown"}).fillna("Unknown")

    if "Specific_Category" in d.columns:
        d["Category_clean"] = d["Specific_Category"].fillna("Unknown").astype(str)
    else:
        d["Category_clean"] = d["Category"].fillna("Unknown").astype(str)

    d["Source File"] = d["Source File"].fillna("Unknown").astype(str)
    d["made_in_clean"] = d["made_in"].apply(clean_country)

    if "exchange_rate" in d.columns:
        d["fx_rate"] = d["exchange_rate"]
    else:
        d["fx_rate"] = d["fx_rate_live_usd_inr"]

    # make on-sale values numeric
    for col in ["US_on_sale", "IN_on_sale"]:
        if col in d.columns:
            d[col] = (
                d[col].astype(str).str.lower()
                .map({"true": 1, "false": 0, "nan": np.nan})
            )

    material_df = d["materials_text"].apply(extract_material_features).apply(pd.Series)
    d = pd.concat([d, material_df], axis=1)

    return d

mango_clean = prep_brand_df(mango, "Mango")
zara_clean = prep_brand_df(zara, "Zara")

wide_df = pd.concat([mango_clean, zara_clean], ignore_index=True)

display(wide_df.head())
print("Combined wide shape:", wide_df.shape)

,Product_ID,Segment,Source File,Specific_Category,US_url,IN_url,name,colour_from_title,US_price_current_usd,US_price_original_usd,US_discount_pct_display,US_discount_pct_calc,US_on_sale,IN_price_current_inr,IN_price_original_inr,IN_price_current_usd,IN_price_original_usd,IN_discount_pct_display,IN_discount_pct_calc,IN_on_sale,Savings_USD,materials_text,made_in,US_detail_selector,US_composition_selector,IN_detail_selector,IN_composition_selector,row_collected_at_local,row_collected_at_utc,fx_rate_live_usd_inr,fx_rate_timestamp_utc,Brand,Category_clean,made_in_clean,fx_rate,cotton_max_pct,has_cotton,polyester_max_pct,has_polyester,linen_max_pct,has_linen,wool_max_pct,has_wool,leather_max_pct,has_leather,viscose_max_pct,has_viscose,lyocell_max_pct,has_lyocell,elastane_max_pct,has_elastane,spandex_max_pct,has_spandex,modal_max_pct,has_modal,silk_max_pct,has_silk,acrylic_max_pct,has_acrylic,polyamide_max_pct,has_polyamide,nylon_max_pct,has_nylon,cashmere_max_pct,has_cashmere,material_char_len,material_piece_count,mentions_composition,Category,date,exchange_rate
0,17001208,Women,comparison_jeans,Women_jeans,https://shop.mango.com/us/en/p/women/jeans/wid...,https://shop.mango.com/in/en/p/women/jeans/wid...,High-rise wide leg jeans with pockets - Women ...,NaN,79.99,NaN,NaN,NaN,0.0,4290.0,NaN,46.40,NaN,NaN,NaN,0.0,33.59,Composition: 100% cotton,Bangladesh\nDye,button:has-text('DETAILS'),NaN,text=DETAILS,NaN,2026-04-10T00:23:43.741040-04:00,2026-04-10T04:23:43.741062+00:00,92.45296,2026-04-10T04:22:00+00:00,Mango,Women_jeans,Bangladesh\nDye,92.45296,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.0,1.0,1.0,NaN,NaN,NaN
1,17001212,Women,comparison_jeans,Women_jeans,https://shop.mango.com/us/en/p/women/jeans/wid...,https://shop.mango.com/in/en/p/women/jeans/wid...,Wide leg pleated jeans - Women | MANGO USA,NaN,69.99,NaN,NaN,NaN,0.0,3990.0,NaN,43.15,NaN,NaN,NaN,0.0,26.84,Composition: 100% cotton | Lining: 65% polyest...,Bangladesh\nDye,text=DETAILS,text=Composition,text=DETAILS,NaN,2026-04-10T00:24:31.366829-04:00,2026-04-10T04:24:31.366850+00:00,92.47754,2026-04-10T04:23:00+00:00,Mango,Women_jeans,Bangladesh\nDye,92.47754,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,3.0,1.0,NaN,NaN,NaN
2,17007792,Women,comparison_knitwear,Women_knitwear,https://shop.mango.com/us/en/p/women/sweaters-...,https://shop.mango.com/in/en/p/women/sweaters-...,Medium-knit sweater - Women | MANGO USA,NaN,69.99,NaN,NaN,NaN,0.0,4190.0,NaN,45.31,NaN,NaN,NaN,0.0,24.68,"Composition: 60% acrylic, 37% polyester, 3% wool",China\nDye,text=DETAILS,NaN,text=DETAILS,NaN,2026-04-10T00:25:02.124197-04:00,2026-04-10T04:25:02.124226+00:00,92.47754,2026-04-10T04:23:00+00:00,Mango,Women_knitwear,China\nDye,92.47754,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,3.0,1.0,NaN,NaN,NaN
3,17007908,Men,comparison_men_knitwear,Men_men_knitwear,https://shop.mango.com/us/en/p/men/sweaters-an...,https://shop.mango.com/in/en/p/men/sweaters-an...,Fine-knit wool-blend sweater - Men | MANGO USA,NaN,99.99,NaN,NaN,NaN,0.0,6290.0,NaN,68.00,NaN,NaN,NaN,0.0,31.99,100% cotton T-shirt regular fit | 100% wool fi...,China\nDye,text=DETAILS,NaN,text=DETAILS,NaN,2026-04-10T00:25:44.611093-04:00,2026-04-10T04:25:44.611135+00:00,92.49570,2026-04-10T04:24:00+00:00,Mango,Men_men_knitwear,China\nDye,92.49570,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,145.0,6.0,1.0,NaN,NaN,NaN
4,17011170,Women,comparison_dresses,Women_dresses,https://shop.mango.com/us/en/p/women/dresses-a...,https://shop.mango.com/in/en/p/women/dresses-a...,Long balloon dress - Women | MANGO USA,NaN,89.99,NaN,NaN,NaN,0.0,4990.0,NaN,53.94,NaN,NaN,NaN,0.0,36.05,"Composition: 70% viscose, 30% polyamide | Lini...",China\nDye,text=DETAILS,NaN,text=

Combined wide shape: (667, 71)


In [9]:
# Make a long table so one row = one product-market pair
long_parts = []

for market in ["US", "IN"]:
    temp = wide_df.copy()
    temp["market"] = market
    temp["price_usd"] = temp[f"{market}_price_current_usd"]
    temp["price_original_usd_market"] = temp.get(f"{market}_price_original_usd", np.nan)
    temp["discount_display_market"] = temp.get(f"{market}_discount_pct_display", np.nan)
    temp["discount_calc_market"] = temp.get(f"{market}_discount_pct_calc", np.nan)
    temp["on_sale_market"] = temp.get(f"{market}_on_sale", np.nan)
    long_parts.append(temp)

long_df = pd.concat(long_parts, ignore_index=True)
long_df = long_df[long_df["price_usd"].notna()].copy()

# add log target for regression
long_df["log_price_usd"] = np.log1p(long_df["price_usd"])

# useful derived fields
long_df["abs_savings_usd"] = long_df["Savings_USD"].abs()
long_df["discount_display_market"] = pd.to_numeric(long_df["discount_display_market"], errors="coerce")
long_df["discount_calc_market"] = pd.to_numeric(long_df["discount_calc_market"], errors="coerce")
long_df["on_sale_market"] = pd.to_numeric(long_df["on_sale_market"], errors="coerce")

print("Long shape:", long_df.shape)
display(long_df[[
    "Brand","Segment","market","Category_clean","price_usd","Savings_USD",
    "discount_display_market","discount_calc_market","made_in_clean"
]].head())

Long shape: (1207, 79)


,Brand,Segment,market,Category_clean,price_usd,Savings_USD,discount_display_market,discount_calc_market,made_in_clean
0,Mango,Women,US,Women_jeans,79.99,33.59,NaN,NaN,Bangladesh\nDye
1,Mango,Women,US,Women_jeans,69.99,26.84,NaN,NaN,Bangladesh\nDye
2,Mango,Women,US,Women_knitwear,69.99,24.68,NaN,NaN,China\nDye
3,Mango,Men,US,Men_men_knitwear,99.99,31.99,NaN,NaN,China\nDye
4,Mango,Women,US,Women_dresses,89.99,36.05,NaN,NaN,China\nDye


In [10]:
# Summary table by brand and segment
summary = (
    long_df.groupby(["Brand", "Segment"])
    .agg(
        rows=("Product_ID", "count"),
        avg_price_usd=("price_usd", "mean"),
        median_price_usd=("price_usd", "median"),
        avg_savings_usd=("Savings_USD", "mean"),
        avg_discount=("discount_calc_market", "mean"),
        avg_fx=("fx_rate", "mean"),
        on_sale_rate=("on_sale_market", "mean")
    )
    .reset_index()
    .sort_values(["Brand", "Segment"])
)

display(summary.style.format({
    "avg_price_usd": "{:.2f}",
    "median_price_usd": "{:.2f}",
    "avg_savings_usd": "{:.2f}",
    "avg_discount": "{:.2f}",
    "avg_fx": "{:.2f}",
    "on_sale_rate": "{:.2%}"
}))

,Brand,Segment,rows,avg_price_usd,median_price_usd,avg_savings_usd,avg_discount,avg_fx,on_sale_rate
0,Mango,Kids,14,33.09,32.86,12.08,nan,92.57,0.00%
1,Mango,Men,48,95.30,79.99,19.64,26.31,92.54,31.25%
2,Mango,Women,86,68.09,55.03,28.18,41.77,92.55,10.47%
3,Zara,Kids,263,32.23,27.35,12.62,58.23,93.52,100.00%
4,Zara,Men,549,81.60,69.90,39.69,62.73,93.62,100.00%
5,Zara,Women,247,55.22,49.90,27.49,28.56,93.66,99.60%


In [11]:
# Graph 1: price distributions by brand, segment, and market
fig = px.violin(
    long_df,
    x="Segment",
    y="price_usd",
    color="Brand",
    facet_col="market",
    box=True,
    points="all",
    hover_data=["name", "Category_clean", "made_in_clean", "Savings_USD"]
)
fig.update_layout(title="Price distribution by Segment, Brand, and Market (USD)")
fig.show()

# Graph 2: average savings heatmap
heat = (
    wide_df.groupby(["Brand", "Segment"])["Savings_USD"]
    .mean()
    .reset_index()
    .pivot(index="Brand", columns="Segment", values="Savings_USD")
)
fig = px.imshow(
    heat,
    text_auto=".2f",
    aspect="auto",
    title="Average US minus India savings by Brand and Segment (USD)"
)
fig.show()

# Graph 3: US vs IN price comparison
paired = wide_df.dropna(subset=["US_price_current_usd", "IN_price_current_usd"]).copy()
fig = px.scatter(
    paired,
    x="US_price_current_usd",
    y="IN_price_current_usd",
    color="Segment",
    symbol="Brand",
    hover_data=["name", "Category_clean", "made_in_clean", "Savings_USD"],
    trendline="ols"
)
fig.update_layout(title="US price vs India price (both in USD)")
fig.show()

# Graph 4: material story
material_cols = [c for c in wide_df.columns if c.startswith("has_")]
material_presence = (
    wide_df.melt(
        id_vars=["Brand", "Segment"],
        value_vars=material_cols,
        var_name="material",
        value_name="present"
    )
)
material_presence = material_presence[material_presence["present"] == 1].copy()
material_presence["material"] = material_presence["material"].str.replace("has_", "", regex=False)

top_materials = (
    material_presence["material"].value_counts().head(10).index.tolist()
)
material_presence = material_presence[material_presence["material"].isin(top_materials)]

fig = px.histogram(
    material_presence,
    x="material",
    color="Brand",
    facet_col="Segment",
    barmode="group"
)
fig.update_layout(title="Top materials by Brand and Segment")
fig.show()

In [12]:
# Regression model to predict price
material_feature_cols = [
    c for c in long_df.columns
    if c.endswith("_max_pct") or c.startswith("has_")
]

numeric_features = [
    "price_original_usd_market",
    "discount_display_market",
    "discount_calc_market",
    "on_sale_market",
    "fx_rate",
    "Savings_USD",
    "abs_savings_usd",
    "material_char_len",
    "material_piece_count",
    "mentions_composition",
] + material_feature_cols

categorical_features = [
    "Brand",
    "Segment",
    "Category_clean",
    "made_in_clean",
    "Source File",
    "market"
]

X = long_df[numeric_features + categorical_features].copy()
y = long_df["log_price_usd"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=10.0))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model.fit(X_train, y_train)

pred_log = model.predict(X_test)
pred_price = np.expm1(pred_log)
actual_price = np.expm1(y_test)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [
        mean_absolute_error(actual_price, pred_price),
        np.sqrt(mean_squared_error(actual_price, pred_price)),
        r2_score(actual_price, pred_price)
    ]
})

display(metrics.style.format({"Value": "{:.4f}"}))

,Metric,Value
0,MAE,13.6475
1,RMSE,33.9206
2,R2,0.6648


In [13]:
# Put predictions back into a dataframe
test_results = X_test.copy()
test_results["actual_price_usd"] = actual_price.values
test_results["predicted_price_usd"] = pred_price
test_results["abs_error"] = (test_results["actual_price_usd"] - test_results["predicted_price_usd"]).abs()

display(
    test_results[[
        "Brand","Segment","market","Category_clean","made_in_clean",
        "actual_price_usd","predicted_price_usd","abs_error"
    ]].head(20)
)

# Graph 5: actual vs predicted
fig = px.scatter(
    test_results,
    x="actual_price_usd",
    y="predicted_price_usd",
    color="Segment",
    symbol="Brand",
    facet_col="market",
    hover_data=["Category_clean", "made_in_clean", "abs_error"],
    trendline="ols"
)
fig.add_shape(
    type="line",
    x0=test_results["actual_price_usd"].min(),
    y0=test_results["actual_price_usd"].min(),
    x1=test_results["actual_price_usd"].max(),
    y1=test_results["actual_price_usd"].max(),
    line=dict(dash="dash")
)
fig.update_layout(title="Actual vs predicted prices")
fig.show()

# Graph 6: prediction error by group
group_error = (
    test_results.groupby(["Brand", "Segment", "market"])
    .agg(
        n=("actual_price_usd", "count"),
        mean_abs_error=("abs_error", "mean")
    )
    .reset_index()
)
fig = px.bar(
    group_error,
    x="Segment",
    y="mean_abs_error",
    color="Brand",
    facet_col="market",
    barmode="group",
    hover_data=["n"]
)
fig.update_layout(title="Mean absolute prediction error by Brand, Segment, and Market")
fig.show()

,Brand,Segment,market,Category_clean,made_in_clean,actual_price_usd,predicted_price_usd,abs_error
101,Zara,Kids,US,GENERAL,China,22.90,37.822297,14.922297
260,Zara,Men,US,GENERAL,Morocco,129.00,119.143888,9.856112
1203,Zara,Men,IN,GENERAL,China,91.33,82.140812,9.189188
109,Zara,Kids,US,GENERAL,Bangladesh,12.90,22.974436,10.074436
649,Zara,Women,US,GENERAL,China,39.90,59.093935,19.193935
736,Mango,Men,IN,Men_men_shirts,China\nDye,53.84,56.536691,2.696691
332,Zara,Men,US,GENERAL,Vietnam,59.90,76.310688,16.410688
49,Mango,Women,US,Women_dresses,China\nDye,139.99,164.254918,24.264918
461,Zara,Men,US,GENERAL,Turkey,99.90,86.437808,13.462192
999,Zara,Men,IN,GENERAL,Vietnam,31.51,23.706721,7.803279


In [14]:
# Top coefficients from the Ridge model
ohe = model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
feature_names = (
    numeric_features
    + list(ohe.get_feature_names_out(categorical_features))
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.named_steps["regressor"].coef_
})

coef_df["abs_coef"] = coef_df["coefficient"].abs()

top_positive = coef_df.sort_values("coefficient", ascending=False).head(15)
top_negative = coef_df.sort_values("coefficient", ascending=True).head(15)
coef_plot = pd.concat([top_negative, top_positive])

fig = px.bar(
    coef_plot.sort_values("coefficient"),
    x="coefficient",
    y="feature",
    orientation="h",
    title="Strongest regression signals pushing predicted price down or up"
)
fig.show()

display(coef_df.sort_values("abs_coef", ascending=False).head(25))

,feature,coefficient,abs_coef
92,Source File_comparison_boytrousers,0.335117,0.335117
95,Source File_comparison_girlnew,-0.332158,0.332158
91,Source File_comparison_boyshirts,-0.316019,0.316019
103,Source File_comparison_outerwear,0.300309,0.300309
115,market_IN,-0.227070,0.227070
116,market_US,0.227070,0.227070
113,Source File_comparison_tshirts,-0.215967,0.215967
6,abs_savings_usd,0.192433,0.192433
96,Source File_comparison_girltops,0.189957,0.189957
42,Segment_Kids,-0.184202,0.184202


In [ ]:
# Optional: save outputs
summary.to_csv("brand_segment_summary.csv", index=False)
test_results.to_csv("predicted_vs_actual_prices.csv", index=False)
coef_df.to_csv("ridge_coefficients.csv", index=False)

print("Saved:")
print("- brand_segment_summary.csv")
print("- predicted_vs_actual_prices.csv")
print("- ridge_coefficients.csv")